# Create MERlin files

Generates every per-experiment MERlin input into `SAMPLE_DIR/merlin/`: the data organization (`dataorganization/`), the analysis-parameters recipe, the snakemake config and the slurm submit script. Codebook and microscope-parameters (YAML) files are NOT copied here: they're shared reference data shipped in `MERci/data/configs/merlin/{codebooks,microscope}/`, and since this `MERci/` clone already lives inside `SAMPLE_DIR/`, the slurm script below references them by their path inside this clone directly — self-contained, no separate cluster-side copy step.

Run this after `create_experiment_info.ipynb` (needs `experiment_info.yaml`).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/create_submit_files/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_26/
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR = SAMPLE_DIR / "settings"
POSITIONS_DIR = SAMPLE_DIR / "positions"
MERLIN_DIR   = SAMPLE_DIR / "merlin"             # new per-experiment MERlin folder
DATA_ORG_DIR = MERLIN_DIR / "dataorganization"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import load_experiment_info, positions_file_tag
from MERci.acquisition.display        import display_file
from MERci.acquisition.data_organization import create_data_organization
from MERci.acquisition.merlin_config import (
    resolve_codebook_filename, resolve_microscope_parameters_filename,
    build_merlin_analysis_parameters,
    create_cluster_resource_allocation, create_snakemake_parameters,
    resolve_cluster_sample_dir, short_experiment_name, create_slurm_submit_script,
)
from MERci.acquisition.pipeline_config import load_pipeline_config

# PIPELINE_ID names this variant's own pipeline.yaml -- see before_imaging notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale. MICROSCOPE/
# OBJECTIVE/LIB_NAME/N_OPT below all come from experiment_info.yaml instead
# (create_experiment_info already threaded them through from this same pipeline.yaml),
# so PIPELINE_CONFIG here is only needed for pipeline-level choices
# (sequential_kind, readout_catalog, the analysis-recipe toggle).
PIPELINE_ID     = "tumor_epi"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / f"{PIPELINE_ID}_pipeline.yaml")

info = load_experiment_info(METADATA_DIR / "experiment_info.yaml")
MICROSCOPE  = info.microscope
OBJECTIVE   = info.extra.get("objective")   # None (older experiment_info.yaml) -> resolve_microscope_parameters_filename falls back to the microscope's default

# SAMPLE_NAME is the TRUE top-level experiment id (from experiment_info.yaml,
# create_experiment_info) -- the same value before_imaging notebooks 01-04 used to name
# positions_*.txt / dave-*.xml / etc., so this notebook's file lookups (and
# its own generated merlin/ filenames) stay consistent with what's already
# on disk.
SAMPLE_NAME = info.sample_name

# SHORT_NAME: SLURM-job-name-safe short form of SAMPLE_NAME (see
# short_experiment_name) -- used below to name this notebook's own
# generated merlin/ filenames instead of the full SAMPLE_NAME, so e.g.
# "merlin_slurm_BC555_sample_05.sh" becomes "merlin_slurm_s5-BC555e.sh".
SHORT_NAME = short_experiment_name(SAMPLE_NAME, info.extra.get("imaging_dir", ""))

# Token positions_path uses below -- must match before_imaging notebook 02's writer, which
# names positions files positions_{POSITIONS_TAG}.txt (POSITIONS_TAG =
# SAMPLE_NAME + "_" + imaging_dir when split into sibling acquisition
# folders, else just SAMPLE_NAME), so sibling split-layout acquisitions of
# the same sample (e.g. tumor/epi vs tumor/disk) never collide on filename.
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, info.extra.get("imaging_dir", ""))

print(f"SAMPLE_NAME : {SAMPLE_NAME}")
print(f"SHORT_NAME  : {SHORT_NAME}")
print(f"MICROSCOPE  : {MICROSCOPE}")
print(f"OBJECTIVE   : {OBJECTIVE}")
print(f"lib_name    : {info.lib_name}")
print(f"data_home   : {info.data_home}")
print(f"merlin_home : {info.merlin_home}")
print(f"folder_name : {info.folder_name}")
print(f"DATA_ORG_DIR : {DATA_ORG_DIR}")

## Data organization

MERlin's per-channel image map, `merlin/dataorganization/data_organization_{MICROSCOPE}_{SAMPLE_NAME}.csv`, built from `round_info.csv`, the frame tables and `round_bit_color_map.csv` (`before_imaging` notebooks 01/03). The sections after it read it back (`generate_mosaic`'s channel count, the slurm script's path).

### Frame tables per round

Each round's frame table comes from `round_info.csv`: its `frame_table`
column when set (a file in `metadata/`), else the table its `hal_config`
was generated with. Rounds imaged with different recipes (e.g. a 650-only
H02 after a 650+560 H01) each get their own. Set a row's `frame_table` in
`round_info.csv` to override it.

In [2]:
from MERci.acquisition.dave import round_frame_tables

CELLS_FT, ROUND_BITS_FT = round_frame_tables(
    pd.read_csv(METADATA_DIR / "round_info.csv"), SETTINGS_DIR, METADATA_DIR)

print(f"Cells frame table : {CELLS_FT.name}")
for hyb_round, path in ROUND_BITS_FT.items():
    ft = pd.read_csv(path, index_col=0)
    colors = sorted(int(c) for c in ft["color"].dropna().unique())
    print(f"Round {hyb_round:>2} frame table: {path.name}  ->  colors: {colors}")

Frame tables found (newest first):
  frame-table-transit-blkf2.csv  →  colors: []
  frame-table-cells-blkf5_405f25_488f2.csv  →  colors: [np.int64(405), np.int64(488)]
  frame-table-bits-blkf5_488f2_560f25_650f25_750f25.csv  →  colors: [np.int64(488), np.int64(560), np.int64(650), np.int64(750)]

Bits  frame table : frame-table-bits-blkf5_488f2_560f25_650f25_750f25.csv
Cells frame table : frame-table-cells-blkf5_405f25_488f2.csv


### Round – bit – color mapping

The round → bit → colour mapping is defined in **before_imaging notebook 03** (which
derives `N_HYBS` from it) and saved to `round_bit_color_map.csv`. Here it is read back for
the data organization. Re-run that notebook to change the mapping.

In [3]:
# round_bit_color is defined & saved in before_imaging notebook 03; read it back here.
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(
        f"{rbc_path} not found — run before_imaging notebook 03 first "
        f"(it now defines the round–bit–color mapping and N_HYBS)."
    )
rbc_df          = pd.read_csv(rbc_path)
round_bit_color = [(int(r), int(b), int(c))
                   for r, b, c in rbc_df[["round", "bit", "color"]].itertuples(index=False, name=None)]
print(f"Loaded {len(round_bit_color)} (round, bit, color) rows from {rbc_path.name}")
print(rbc_df.to_string(index=False))

Loaded 27 (round, bit, color) rows from round_bit_color_map.csv
 round  bit  color
     1    1    750
     1    2    650
     1   17    560
     2    3    750
     2    4    650
     2   18    560
     3    5    750
     3    6    650
     3   19    560
     4    7    750
     4    8    650
     4   20    560
     5    9    750
     5   10    650
     5   21    560
     6   11    750
     6   12    650
     6   22    560
     7   13    750
     7   14    650
     7   23    560
     8   15    750
     8   16    650
     8   24    560
     9   25    750
     9   26    650
     9   27    560


### Series patterns from round_info.csv

Reads the bits and cells series patterns so the correct `imageType` and
`imageRegExp` fields are written into the data organization.  
Re-run before_imaging notebook 03 first if `round_info.csv` is out of date.

In [4]:
round_info_path = METADATA_DIR / "round_info.csv"
round_info      = pd.read_csv(round_info_path)

# Multi-boundary round_info carries per-segment rows (imaging_type in
# {cells, bits, transit} + a 'segment' column). Pick the bits/cells series by
# imaging_type so transit movies are never selected; fall back to name matching
# for the legacy schema.
MULTI_BOUNDARY = "segment" in round_info.columns
if "imaging_type" in round_info.columns:
    itype        = round_info["imaging_type"].astype(str).str.lower()
    bits_rows    = round_info[itype == "bits"]
    cells_rows   = round_info[itype == "cells"]
else:
    bits_rows    = round_info[~round_info["series"].str.contains("cells")]
    cells_rows   = round_info[ round_info["series"].str.contains("cells")]

bits_series  = bits_rows.iloc[0]["series"]
cells_series = cells_rows.iloc[0]["series"]

print(f"Multi-boundary layout : {MULTI_BOUNDARY}")
print(f"Bits  series (sample) : {bits_series}")
print(f"Cells series (sample) : {cells_series}")

if MULTI_BOUNDARY:
    tissues = sorted(round_info["tissue"].dropna().astype(int).unique())
    print(f"\nTissues present       : {tissues}")
    print("NOTE: multi-tissue MERlin analysis is per tissue, and each boundary is a\n"
          "distinct movie (imageType). The cell below builds ONE data-organization from\n"
          "the representative series above; for a per-tissue / per-boundary MERlin run,\n"
          "confirm the intended workflow before relying on this file.")

Multi-boundary layout : False
Bits  series (sample) : hal-mf4-epi_01_{fov:03d}
Cells series (sample) : hal-mf4-epi-cells_{fov:03d}


### Sequential-bit gene names

Non-barcode bit -> target-gene-name mapping is **pipeline-level**: read
from `lib_name`'s own `<codebook>_{sequential_kind}.csv` companion file in
`data/configs/merlin/codebooks/` (`sequential_kind` is this pipeline's own
choice -- "sequential" for an RNA panel, "immuno" for an antibody panel;
empty if this library has no such bits). If `metadata/sequential_genes.csv`
also exists for this experiment, its rows override the codebook's defaults.
Either way, the resulting mapping is used as the `channelName` for those
bits in the data organization (instead of the generic `bitNN` label), so
downstream tasks (`SmfishSignal`/`SumSignal`) can reference bits by target
name.

In [ ]:
from MERci.acquisition.merlin_config import load_sequential_gene_names

CODEBOOKS_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "codebooks"
sequential_genes_path = METADATA_DIR / "sequential_genes.csv"   # per-experiment override, optional
sequential_gene_names = load_sequential_gene_names(
    CODEBOOKS_DIR, PIPELINE_CONFIG.merlin.lib_name, sequential_genes_path,
    kind=PIPELINE_CONFIG.merlin.sequential_kind,
) or None   # create_data_organization treats {} the same as None -- keep the explicit fallback for clarity

if sequential_gene_names:
    print(f"Loaded {len(sequential_gene_names)} target gene name(s) for lib_name={PIPELINE_CONFIG.merlin.lib_name!r} "
          f"(kind={PIPELINE_CONFIG.merlin.sequential_kind!r})"
          + (f" (+ per-experiment override {sequential_genes_path.name})" if sequential_genes_path.exists() else "") + ":")
    for bit, gene in sorted(sequential_gene_names.items()):
        print(f"  bit {bit}: {gene}")
else:
    print(f"No target gene names for lib_name={PIPELINE_CONFIG.merlin.lib_name!r} "
          f"(kind={PIPELINE_CONFIG.merlin.sequential_kind!r}) -- bits (if any) will use default bitNN channelNames.")

### Readout-name overrides (adaptor-based channels)

By default a bit's `readoutName` comes from `data/readouts.csv`
(`Bit number` -> `Probe name`, the RS-series readouts). Two ways to
override it:

- `dataorganization.readout_catalog: ndb` in pipeline.yaml: every bit
  number is an NDB adaptor number, so bit 865 reads out `NDB_865`
  (`data/ndb_adaptors.csv`).
- An optional per-experiment `metadata/readout_adaptors.csv` (columns
  `bit,probe_name`), which wins over both.

Every override name is checked against `readouts.csv`/`ndb_adaptors.csv`.
MERlin never reads `bitNumber`; only `readoutName` is matched against the
codebook's `bit_names`, so a non-RS name can't be mistaken for a barcode
bit.

In [ ]:
from MERci.acquisition.merlin_config import load_readout_name_overrides

adaptors = pd.read_csv(MERCI_DIR / "data" / "ndb_adaptors.csv")
readout_adaptors_path = METADATA_DIR / "readout_adaptors.csv"   # per-experiment, optional
readout_name_overrides = load_readout_name_overrides(readout_adaptors_path) or None
# create_data_organization treats {} the same as None -- keep the explicit fallback for clarity

# readout_catalog "ndb" (pipeline.yaml): each bit number is its NDB adaptor
# number, so its readoutName is NDB_<bit>. A per-experiment
# readout_adaptors.csv entry still wins for its bit.
if PIPELINE_CONFIG.readout_catalog == "ndb":
    readout_name_overrides = {**{bit: f"NDB_{bit}" for _, bit, _ in round_bit_color},
                              **(readout_name_overrides or {})}

if readout_name_overrides:
    print(f"{len(readout_name_overrides)} readoutName override(s) (readout_catalog={PIPELINE_CONFIG.readout_catalog!r}"
          + (f", + {readout_adaptors_path.name}" if readout_adaptors_path.exists() else "") + "):")
    for bit, probe_name in sorted(readout_name_overrides.items()):
        print(f"  bit {bit}: {probe_name}")
else:
    print(f"No readoutName overrides -- all bits use readouts.csv's default Bit number -> Probe name lookup.")

### Build and save data organization

In [ ]:
readouts  = pd.read_csv(MERCI_DIR / "data" / "readouts.csv")
ft_bits   = {r: pd.read_csv(path, index_col=0) for r, path in ROUND_BITS_FT.items()}
ft_cells  = pd.read_csv(CELLS_FT, index_col=0)

data_org = create_data_organization(
    bits_frame_table  = ft_bits,
    cells_frame_table = ft_cells,
    round_bit_color   = round_bit_color,
    readouts          = readouts,
    bits_series       = bits_series,
    cells_series      = cells_series,
    include_dapi      = True,
    sequential_gene_names = sequential_gene_names,
    readout_name_overrides = readout_name_overrides,
    adaptors          = adaptors,
)

data_org_path = DATA_ORG_DIR / f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
data_org_path.parent.mkdir(parents=True, exist_ok=True)
data_org.to_csv(data_org_path, index=False)
print(f"Saved: {data_org_path}")
print(f"\n{len(data_org)} rows  ({len(data_org)-1} bits + DAPI)")
data_org

## Resolve shared reference files (codebook, microscope parameters)

These live in `MERci/data/configs/merlin/` — shipped with the repo, not regenerated per experiment. As a sanity check, the codebook's own `bit_names` row is compared against this experiment's bit count from `round_bit_color_map.csv`, to catch a mismatched codebook before it reaches the cluster.

In [ ]:
from MERci.acquisition.merlin_config import load_sequential_gene_names, resolve_sequential_codebook_filename

codebook_path   = MERCI_DIR / "data" / "configs" / "merlin" / "codebooks" / resolve_codebook_filename(info.lib_name)
microscope_path = MERCI_DIR / "data" / "configs" / "merlin" / "microscope" / resolve_microscope_parameters_filename(MICROSCOPE, OBJECTIVE)

for p in (codebook_path, microscope_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found -- add it to MERci/data/configs/merlin/ before running this notebook."
        )

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
sequential_genes_path = METADATA_DIR / "sequential_genes.csv"   # per-experiment override, optional
SEQUENTIAL_KIND = PIPELINE_CONFIG.merlin.sequential_kind
# Non-barcode (sequential/immuno) target names aren't in the codebook by
# design -- pipeline-level, from lib_name's own <codebook>_<SEQUENTIAL_KIND>.csv
# companion (load_sequential_gene_names), not a per-experiment file, so this
# bit-count check coincides with the codebook + its companion regardless of
# whether this experiment's metadata/ has its own sequential_genes.csv.
# Restricted to the bits this experiment actually imaged (round_bit_color_map.csv),
# so a companion bit it never imaged can't add (or repeat) a channel name.
imaged_bits = pd.read_csv(rbc_path)["bit"].tolist() if rbc_path.exists() else None
sequential_gene_names = load_sequential_gene_names(codebook_path.parent, info.lib_name, sequential_genes_path,
                                                   kind=SEQUENTIAL_KIND, imaged_bits=imaged_bits)
n_sequential_bits = len(sequential_gene_names)

# Pipelines with no barcode-decode task (pipeline.yaml's merlin.analysis.tasks)
# never image a combinatorial barcode -- the codebook's own bit_names count
# has nothing to reconcile against round_bit_color then; lib_name is still
# used for the codebook/microscope-parameters file paths above and for this
# pipeline's own SEQUENTIAL_KIND companion, just not for a barcode-count check.
DOES_DECODE = PIPELINE_CONFIG.merlin.tasks.get("decode", False)
if not DOES_DECODE:
    n_bits = int(pd.read_csv(rbc_path)["bit"].nunique()) if rbc_path.exists() else None
    print(f"No barcode-decode task in this pipeline's recipe -- skipping the codebook "
          f"bit-count check. {n_sequential_bits} {SEQUENTIAL_KIND} target(s) from "
          f"{resolve_sequential_codebook_filename(info.lib_name, SEQUENTIAL_KIND)}.")
elif rbc_path.exists():
    # Distinct bit count (NOT round.max() -- a round can carry several bits,
    # one per color, so round.max() is the hyb-round count, not the bit count).
    n_bits = int(pd.read_csv(rbc_path)["bit"].nunique())
    n_barcode_bits = n_bits - n_sequential_bits
    codebook_bit_names_line = next(
        line for line in codebook_path.read_text().splitlines() if line.startswith("bit_names")
    )
    codebook_n_bits = len(codebook_bit_names_line.split(",")) - 1
    if codebook_n_bits != n_barcode_bits:
        print(f"WARNING: codebook {codebook_path.name} has {codebook_n_bits} bits, "
              f"but this experiment images {n_barcode_bits} barcode bits (of {n_bits} total) "
              f"-- double-check lib_name/codebook choice.")
    else:
        print(f"Bit count OK: {n_barcode_bits} barcode bits (of {n_bits} total imaged, "
              f"{n_sequential_bits} {SEQUENTIAL_KIND}), matching {codebook_path.name}"
              + (f" + {resolve_sequential_codebook_filename(info.lib_name, SEQUENTIAL_KIND)}" if n_sequential_bits else "") + ".")
else:
    n_bits = None
    print(f"WARNING: {rbc_path} not found -- run before_imaging notebook 03 first.")

print(f"Codebook  : {codebook_path}")
print(f"Microscope: {microscope_path}")

## MERlin analysis-parameters JSON

Built from an explicit recipe (`data/configs/merlin/analysis/recipes/`) of atomic MERlin tasks (`data/configs/merlin/analysis/tasks/`) instead of copying and hand-editing a prior experiment's file -- see `build_merlin_analysis_parameters`'s docstring for the recipe/atom format, and each `.yaml` file directly to see/tune its parameters.

In [ ]:
# sequential_gene_names (pipeline-level codebook_sequential/immuno + optional
# per-experiment override) was already resolved in the cell above.
sequential_channel_names = [sequential_gene_names[b] for b in sorted(sequential_gene_names)]
if sequential_channel_names:
    print(f"Target genes (bit order): {sequential_channel_names}")

# The pipeline's own task menu (pipeline.yaml's merlin.analysis.tasks) is the
# source of truth for which atoms run -- written out below as this
# experiment's own recipe file, so it always matches pipeline.yaml exactly.
merlin_cfg  = PIPELINE_CONFIG.merlin
overrides   = dict(merlin_cfg.overrides)
extra_tasks = []

# smfish_signal needs real channel_names, resolved per experiment from the
# codebook's sequential/immuno companion (optionally overridden via
# metadata/sequential_genes.csv) -- pipeline.yaml has no sane static default
# for these. Supply the override whenever there's real content; only append
# the task itself if pipeline.yaml hasn't already enabled it, to avoid a
# duplicate task entry (build_merlin_analysis_parameters concatenates the
# pipeline's own task list with extra_tasks, with no deduplication).
if sequential_channel_names:
    overrides["smfish_signal"] = {"channel_names": sequential_channel_names}
    if "smfish_signal" not in merlin_cfg.enabled_tasks:
        extra_tasks.append("smfish_signal")

# generate_mosaic needs real z_index/data_channels, resolved per experiment
# the same way smfish_signal's channel_names are above -- pipeline.yaml has
# no sane static default for either (MERlin's GenerateMosaicTile has no
# "export everything" default any more). data_channels: every data channel
# in this experiment's own data organization (above; one row per
# channel, so "all channels" is just its row count). z_index: the middle
# index of this pipeline's z-stack (cells_round/bits_round share the same
# z_min/z_max/z_step in every pipeline that enables generate_mosaic).
if "generate_mosaic" in merlin_cfg.enabled_tasks:
    n_data_channels = len(data_org)   # built in "Data organization" above
    z_recipe = PIPELINE_CONFIG.bits_round
    n_z = round((z_recipe.z_max - z_recipe.z_min) / z_recipe.z_step) + 1
    mosaic_z_index = n_z // 2
    overrides["generate_mosaic"] = {
        "z_index": mosaic_z_index,
        "data_channels": list(range(n_data_channels)),
    }
    print(f"generate_mosaic: z_index={mosaic_z_index} (of {n_z}), data_channels=all {n_data_channels}")

ANALYSIS_TASKS_DIR   = MERCI_DIR / "data" / "configs" / "merlin" / "analysis" / "tasks"
N_OPTIMIZE_ITERATIONS = info.extra.get("n_opt", 10)   # MERlin param, independent of bit count; reused below (snakemake cell)

# This experiment's own recipe, generated from pipeline.yaml's task menu --
# same {n_optimize_iterations, tasks} shape as the shared recipe files, so
# build_merlin_analysis_parameters needs no changes.
generated_recipe_path = MERLIN_DIR / "analysis" / f"recipe_{SHORT_NAME}.yaml"
generated_recipe_path.parent.mkdir(parents=True, exist_ok=True)
import yaml as _yaml
with open(generated_recipe_path, "w") as fh:
    _yaml.safe_dump({"n_optimize_iterations": N_OPTIMIZE_ITERATIONS, "tasks": merlin_cfg.enabled_tasks}, fh, sort_keys=False)
print(f"Generated recipe (from pipeline.yaml): {generated_recipe_path}")

analysis_path = MERLIN_DIR / "analysis" / f"merlin_analysis_{SHORT_NAME}.yaml"
build_merlin_analysis_parameters(
    recipe_path            = generated_recipe_path,
    tasks_dir               = ANALYSIS_TASKS_DIR,
    output_path              = analysis_path,
    n_optimize_iterations     = N_OPTIMIZE_ITERATIONS,
    extra_tasks                = extra_tasks,
    overrides                   = overrides,
)
print(f"Saved: {analysis_path}")
display_file(analysis_path)

## Snakemake cluster-resource-allocation + parameters YAML

In [ ]:
cluster_template = MERCI_DIR / "data" / "configs" / "merlin" / "snakemake" / "cluster_resource_allocation_basic.json"

cluster_resource_path = MERLIN_DIR / "snakemake" / f"cluster_resource_allocation_{SHORT_NAME}.yaml"
create_cluster_resource_allocation(
    template_path=cluster_template, exp_name=SAMPLE_NAME,
    n_optimize_iterations=N_OPTIMIZE_ITERATIONS, output_path=cluster_resource_path,
)
print(f"Saved: {cluster_resource_path}")
display_file(cluster_resource_path)

snakemake_params_path = MERLIN_DIR / "snakemake" / f"parameters_{SHORT_NAME}.yaml"
create_snakemake_parameters(
    exp_name=SAMPLE_NAME, cluster_config_path=cluster_resource_path,
    output_path=snakemake_params_path,
    job_name_prefix=SHORT_NAME,
)
print(f"Saved: {snakemake_params_path}")
display_file(snakemake_params_path)

## Slurm submit script

Every path in the generated script is written relative to one `$SAMPLE_DIR` bash variable (this experiment's acquisition root — see `resolve_cluster_sample_dir`), instead of five separately-resolved absolute paths. This makes the script identical whether it was generated here (Windows, before transferring `SAMPLE_DIR` to the cluster — `$SAMPLE_DIR` is then *predicted* from the sample name) or regenerated on the cluster itself after the transfer (Linux — `$SAMPLE_DIR` is just the real current path, no guessing).

In [ ]:
positions_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"

# Use the camera-rotation-corrected positions file if notebooks/misc/
# correct_camera_rotation.ipynb has been run for this experiment -- MERlin
# should always decode against the corrected FOV positions when available,
# since the raw grid assumes zero camera-vs-stage rotation.
corrected_positions_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_corrected.txt"
if corrected_positions_path.exists():
    positions_path = corrected_positions_path
    print(f"Using camera-rotation-corrected positions file: {positions_path.name} "
          f"(see notebooks/misc/correct_camera_rotation.ipynb)")

for p in (data_org_path, positions_path):
    if not p.exists():
        print(f"WARNING: {p} not found. For a multi-boundary experiment, set the "
              f"correct per-segment file path manually before submitting to the cluster.")

# $SAMPLE_DIR in the generated script: this experiment's acquisition root as it
# will be addressed from the Linux cluster. Predicted from the TRUE sample_name +
# imaging_dir (experiment_info.yaml, create_experiment_info) when running on Windows
# (before transfer); the real current path when already running on the
# cluster (Linux) -- see resolve_cluster_sample_dir.
sample_dir_cluster = resolve_cluster_sample_dir(SAMPLE_DIR, SAMPLE_NAME, info.extra.get("imaging_dir", ""))

def _rel(p):
    """POSIX path relative to SAMPLE_DIR, for use as "$SAMPLE_DIR/<...>" in the script."""
    return Path(p).relative_to(SAMPLE_DIR).as_posix()

submit_path = MERLIN_DIR / "slurm" / "submit" / f"merlin_slurm_{SHORT_NAME}.sh"
create_slurm_submit_script(
    label                   = SHORT_NAME,
    sample_dir              = sample_dir_cluster,
    parameters_file         = _rel(snakemake_params_path),
    analysis_file           = _rel(analysis_path),
    data_organization_file  = _rel(data_org_path),
    positions_file          = _rel(positions_path),
    codebook_file           = _rel(codebook_path),
    microscope_file         = _rel(microscope_path),
    data_home               = info.data_home,
    folder_name             = info.folder_name,
    output_path             = submit_path,
    analysis_name           = "output",
)
print(f"$SAMPLE_DIR (cluster) : {sample_dir_cluster}")
print(f"Saved: {submit_path}\n")
display_file(submit_path)